In [1]:
from turbofan_fc_cnn_mc import fc_cnn_mc
from util.postProcess import post
from util import prelim
import numpy as np
from saveasmat_mc import saveasmat_mc
import time

In [2]:
database = '../database2/BU_supaero_10_64_66_0.07m_0.16m_conv.json'

## Construct training data

In [3]:
rotor_test = [19, 24, 46] # Geometries to be used for testing.
rotor_drop = [] # Geometries to be removed from dataset.
input_keys = ["M_tip", "phi","th_le","th_te","x_le","x_te","radius_le","radius_te","dist_le","dist_te",
              "chord","stagger","metal_angle_le","metal_angle_te","thickness_upper_1","thickness_upper_2","thickness_upper_3",
             "thickness_upper_4","thickness_upper_5","thickness_upper_6","thickness_lower_1","thickness_lower_2",
             "thickness_lower_3","thickness_lower_4","thickness_lower_5","thickness_lower_6"] 
output_keys = ["tke", "LenScale", "vx", "vt", "p", "T", "v_mag"]
COORD_KEYS = ["x", "y", "z", "r", "th"]
input_, output_, coordinate, label = prelim.load_data_multiChan(
    file=database,
    input_keys=input_keys,
    output_keys=output_keys,total=True,metric=True)
evalSlice = prelim.find_slice_by_geom_and_slice(label,geom_values=rotor_test, rpm_values = [7000,9000,12000])
dropSlice = prelim.find_slice_by_geom_and_slice(label,geom_values=rotor_test, rpm_values = [15000])

input shape      : (2890, 122)
output shape     : (2890, 64, 64, 7)
coordinate shape : (2890, 64, 64, 5)


In [4]:
allSlice = range(input_.shape[0])
trainSlice = [i for i in allSlice if i not in set(evalSlice + dropSlice)]

In [6]:
## Train/test spliting
reg_input_train, reg_input_test, reg_output_train, reg_output_test = prelim.regularized_train_test_mc(input_, output_, evalSlice, trainSlice=trainSlice, norm_path="normalize_parameters.pkl")
label_train, coordinate_train, label_test, coordinate_test = prelim.split_coordinate_label(coordinate, label, evalSlice, trainSlice=trainSlice)

## Train multi-output model

In [7]:
start_time = time.perf_counter()

mc_model = fc_cnn_mc(reg_input_train,reg_output_train)
#mc_model = fc_cnn_mc(reg_input_train,reg_output_train,reg_input_test,reg_output_test)
#mc_model.build_model(print_summary = True,lr=[0.002,1000,0.985])
mc_model.build_model(print_summary = True,lr=[0.0012,1000,0.98])
mc_model.fit_model(ep=3000,p=120,ts=0.1,bs=32)
mc_model.model.save('Models/mc.h5')

end_time = time.perf_counter()
elapsed = end_time - start_time
saveasmat_mc("Results/", mc_model.model,reg_input_train,reg_input_test,reg_output_train,reg_output_test,label_train,coordinate_train,label_test,coordinate_test,nfile='normalize_parameters_mc.pkl',output_keys=output_keys)

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 256)               31488     
                                                                 
 dense_1 (Dense)             (None, 128)               32896     
                                                                 
 dense_2 (Dense)             (None, 64)                8256      
                                                                 
 reshape (Reshape)           (None, 8, 8, 1)           0         
                                                                 
 conv2d_transpose (Conv2DTra  (None, 8, 8, 256)        2560      
 nspose)                                                         
                                                                 
 leaky_re_lu (LeakyReLU)     (None, 8, 8, 256)         0         
                                                        

Epoch 96/3000
64/64 [==============================] - 0s 4ms/step - loss: 0.0323 - mae: 0.1082 - val_loss: 0.0288 - val_mae: 0.1145
Epoch 97/3000
64/64 [==============================] - 0s 4ms/step - loss: 0.0359 - mae: 0.1183 - val_loss: 0.0374 - val_mae: 0.1211
Epoch 98/3000
64/64 [==============================] - 0s 4ms/step - loss: 0.0331 - mae: 0.1129 - val_loss: 0.0232 - val_mae: 0.0995
Epoch 99/3000
64/64 [==============================] - 0s 4ms/step - loss: 0.0279 - mae: 0.1028 - val_loss: 0.0443 - val_mae: 0.1356
Epoch 100/3000
64/64 [==============================] - 0s 4ms/step - loss: 0.0406 - mae: 0.1243 - val_loss: 0.0365 - val_mae: 0.1242
Epoch 101/3000
64/64 [==============================] - 0s 4ms/step - loss: 0.0301 - mae: 0.1040 - val_loss: 0.0227 - val_mae: 0.0925
Epoch 102/3000
64/64 [==============================] - 0s 4ms/step - loss: 0.0239 - mae: 0.0926 - val_loss: 0.0230 - val_mae: 0.0965
Epoch 103/3000
64/64 [==============================] - 0s 4ms/ste

64/64 [==============================] - 0s 4ms/step - loss: 0.0106 - mae: 0.0667 - val_loss: 0.0091 - val_mae: 0.0627
Epoch 219/3000
64/64 [==============================] - 0s 4ms/step - loss: 0.0117 - mae: 0.0644 - val_loss: 0.0206 - val_mae: 0.0926
Epoch 220/3000
64/64 [==============================] - 0s 4ms/step - loss: 0.0242 - mae: 0.0946 - val_loss: 0.0227 - val_mae: 0.0970
Epoch 221/3000
64/64 [==============================] - 0s 4ms/step - loss: 0.0170 - mae: 0.0809 - val_loss: 0.0093 - val_mae: 0.0639
Epoch 222/3000
64/64 [==============================] - 0s 4ms/step - loss: 0.0141 - mae: 0.0717 - val_loss: 0.0085 - val_mae: 0.0612
Epoch 223/3000
64/64 [==============================] - 0s 4ms/step - loss: 0.0251 - mae: 0.0910 - val_loss: 0.0249 - val_mae: 0.1089
Epoch 224/3000
64/64 [==============================] - 0s 4ms/step - loss: 0.0159 - mae: 0.0792 - val_loss: 0.0099 - val_mae: 0.0653
Epoch 225/3000
64/64 [==============================] - 0s 4ms/step - loss: 0

64/64 [==============================] - 0s 4ms/step - loss: 0.0165 - mae: 0.0761 - val_loss: 0.0119 - val_mae: 0.0728
Epoch 341/3000
64/64 [==============================] - 0s 4ms/step - loss: 0.0119 - mae: 0.0687 - val_loss: 0.0078 - val_mae: 0.0578
Epoch 342/3000
64/64 [==============================] - 0s 4ms/step - loss: 0.0173 - mae: 0.0751 - val_loss: 0.0141 - val_mae: 0.0767
Epoch 343/3000
64/64 [==============================] - 0s 4ms/step - loss: 0.0161 - mae: 0.0768 - val_loss: 0.0086 - val_mae: 0.0613
Epoch 344/3000
64/64 [==============================] - 0s 4ms/step - loss: 0.0072 - mae: 0.0540 - val_loss: 0.0051 - val_mae: 0.0472
Epoch 345/3000
64/64 [==============================] - 0s 4ms/step - loss: 0.0055 - mae: 0.0472 - val_loss: 0.0049 - val_mae: 0.0471
Epoch 346/3000
64/64 [==============================] - 0s 4ms/step - loss: 0.0057 - mae: 0.0483 - val_loss: 0.0045 - val_mae: 0.0449
Epoch 347/3000
64/64 [==============================] - 0s 4ms/step - loss: 0

64/64 [==============================] - 0s 4ms/step - loss: 0.0047 - mae: 0.0446 - val_loss: 0.0040 - val_mae: 0.0434
Epoch 463/3000
64/64 [==============================] - 0s 4ms/step - loss: 0.0049 - mae: 0.0461 - val_loss: 0.0036 - val_mae: 0.0413
Epoch 464/3000
64/64 [==============================] - 0s 4ms/step - loss: 0.0044 - mae: 0.0427 - val_loss: 0.0036 - val_mae: 0.0404
Epoch 465/3000
64/64 [==============================] - 0s 4ms/step - loss: 0.0047 - mae: 0.0434 - val_loss: 0.0051 - val_mae: 0.0479
Epoch 466/3000
64/64 [==============================] - 0s 4ms/step - loss: 0.0047 - mae: 0.0444 - val_loss: 0.0039 - val_mae: 0.0421
Epoch 467/3000
64/64 [==============================] - 0s 4ms/step - loss: 0.0045 - mae: 0.0438 - val_loss: 0.0047 - val_mae: 0.0454
Epoch 468/3000
64/64 [==============================] - 0s 4ms/step - loss: 0.0102 - mae: 0.0606 - val_loss: 0.0092 - val_mae: 0.0598
Epoch 469/3000
64/64 [==============================] - 0s 4ms/step - loss: 0

64/64 [==============================] - 0s 4ms/step - loss: 0.0038 - mae: 0.0412 - val_loss: 0.0028 - val_mae: 0.0358
Epoch 585/3000
64/64 [==============================] - 0s 4ms/step - loss: 0.0036 - mae: 0.0391 - val_loss: 0.0030 - val_mae: 0.0363
Epoch 586/3000
64/64 [==============================] - 0s 4ms/step - loss: 0.0037 - mae: 0.0387 - val_loss: 0.0046 - val_mae: 0.0452
Epoch 587/3000
64/64 [==============================] - 0s 4ms/step - loss: 0.0057 - mae: 0.0491 - val_loss: 0.0048 - val_mae: 0.0465
Epoch 588/3000
64/64 [==============================] - 0s 4ms/step - loss: 0.0086 - mae: 0.0583 - val_loss: 0.0081 - val_mae: 0.0574
Epoch 589/3000
64/64 [==============================] - 0s 4ms/step - loss: 0.0054 - mae: 0.0470 - val_loss: 0.0034 - val_mae: 0.0387
Epoch 590/3000
64/64 [==============================] - 0s 4ms/step - loss: 0.0035 - mae: 0.0383 - val_loss: 0.0028 - val_mae: 0.0353
Epoch 591/3000
64/64 [==============================] - 0s 4ms/step - loss: 0

... (23 outputs left out)

64/64 [==============================] - 0s 4ms/step - loss: 0.0010 - mae: 0.0211 - val_loss: 8.9401e-04 - val_mae: 0.0205
Epoch 2853/3000
64/64 [==============================] - 0s 4ms/step - loss: 0.0010 - mae: 0.0211 - val_loss: 8.9279e-04 - val_mae: 0.0205
Epoch 2854/3000
64/64 [==============================] - 0s 4ms/step - loss: 0.0010 - mae: 0.0211 - val_loss: 8.9641e-04 - val_mae: 0.0206
Epoch 2855/3000
64/64 [==============================] - 0s 4ms/step - loss: 0.0010 - mae: 0.0211 - val_loss: 8.9310e-04 - val_mae: 0.0205
Epoch 2856/3000
64/64 [==============================] - 0s 4ms/step - loss: 0.0010 - mae: 0.0211 - val_loss: 8.9628e-04 - val_mae: 0.0205
Epoch 2857/3000
64/64 [==============================] - 0s 4ms/step - loss: 0.0010 - mae: 0.0211 - val_loss: 8.9585e-04 - val_mae: 0.0205
Epoch 2858/3000
64/64 [==============================] - 0s 4ms/step - loss: 0.0010 - mae: 0.0211 - val_loss: 8.9737e-04 - val_mae: 0.0205
Epoch 2859/3000
64/64 [====================

64/64 [==============================] - 0s 4ms/step - loss: 0.0010 - mae: 0.0210 - val_loss: 8.8786e-04 - val_mae: 0.0204
Epoch 2912/3000
64/64 [==============================] - 0s 4ms/step - loss: 0.0010 - mae: 0.0210 - val_loss: 8.8679e-04 - val_mae: 0.0204
Epoch 2913/3000
64/64 [==============================] - 0s 4ms/step - loss: 0.0010 - mae: 0.0210 - val_loss: 8.8784e-04 - val_mae: 0.0204
Epoch 2914/3000
64/64 [==============================] - 0s 4ms/step - loss: 0.0010 - mae: 0.0210 - val_loss: 8.8480e-04 - val_mae: 0.0204
Epoch 2915/3000
64/64 [==============================] - 0s 4ms/step - loss: 0.0010 - mae: 0.0210 - val_loss: 8.8652e-04 - val_mae: 0.0204
Epoch 2916/3000
64/64 [==============================] - 0s 4ms/step - loss: 0.0010 - mae: 0.0210 - val_loss: 8.9036e-04 - val_mae: 0.0204
Epoch 2917/3000
64/64 [==============================] - 0s 4ms/step - loss: 0.0010 - mae: 0.0210 - val_loss: 8.8620e-04 - val_mae: 0.0204
Epoch 2918/3000
64/64 [====================

64/64 [==============================] - 0s 4ms/step - loss: 0.0010 - mae: 0.0210 - val_loss: 8.8917e-04 - val_mae: 0.0204
Epoch 2971/3000
64/64 [==============================] - 0s 4ms/step - loss: 0.0010 - mae: 0.0210 - val_loss: 8.8352e-04 - val_mae: 0.0204
Epoch 2972/3000
64/64 [==============================] - 0s 4ms/step - loss: 0.0010 - mae: 0.0209 - val_loss: 8.8311e-04 - val_mae: 0.0204
Epoch 2973/3000
64/64 [==============================] - 0s 4ms/step - loss: 0.0010 - mae: 0.0209 - val_loss: 8.8107e-04 - val_mae: 0.0203
Epoch 2974/3000
64/64 [==============================] - 0s 4ms/step - loss: 0.0010 - mae: 0.0209 - val_loss: 8.8518e-04 - val_mae: 0.0204
Epoch 2975/3000
64/64 [==============================] - 0s 4ms/step - loss: 0.0010 - mae: 0.0210 - val_loss: 8.8554e-04 - val_mae: 0.0204
Epoch 2976/3000
64/64 [==============================] - 0s 4ms/step - loss: 0.0010 - mae: 0.0210 - val_loss: 8.8739e-04 - val_mae: 0.0204
Epoch 2977/3000
64/64 [====================

FileNotFoundError: [Errno 2] No such file or directory: 'normalize_parameters_mc.pkl'

In [ ]:
print("time elapsed", elapsed, "s") 

## Plot model loss and MAE

In [ ]:
mc_model.plot_loss()
post(mc_model.model,reg_input_test,reg_output_test).pixelError_mc()
post(mc_model.model,reg_input_test,reg_output_test).circError_mc()